## 2026-27 Live Title Simulation

---
## Step 1 - Imports & Setup

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import soccerdata as sd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss, accuracy_score

warnings.filterwarnings('ignore')
%matplotlib inline

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

TITLE_TEAMS = ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']
BIG6 = ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United', 'Chelsea', 'Tottenham']

RIVALRIES = {
    'Arsenal': ['Tottenham', 'Manchester United', 'Chelsea', 'Manchester City'],
    'Brentford': ['Fulham', 'Chelsea'],
    'Everton': ['Liverpool'],
    'Hull': ['Leeds'],
    'Ipswich': [],  # Main rival (Norwich City) is not in the Premier League this season
    'Nottingham Forest': ['Coventry', 'Aston Villa'],
    'Brighton': ['Crystal Palace', 'Bournemouth'],
    'Manchester City': ['Manchester United', 'Liverpool', 'Arsenal'],
    'Newcastle United': ['Sunderland'],
    'Fulham': ['Chelsea', 'Brentford'],
    'Crystal Palace': ['Brighton'],
    'Bournemouth': ['Brighton'],
    'Coventry': ['Aston Villa', 'Nottingham Forest'],
    'Liverpool': ['Manchester United', 'Everton', 'Manchester City', 'Chelsea'],
    'Tottenham': ['Arsenal', 'Chelsea'],
    'Chelsea': ['Arsenal', 'Tottenham', 'Fulham', 'Leeds', 'Liverpool'],
    'Leeds': ['Manchester United', 'Chelsea', 'Hull'],
    'Manchester United': ['Liverpool', 'Manchester City', 'Leeds', 'Arsenal'],
    'Sunderland': ['Newcastle United'],
    'Aston Villa': ['Coventry', 'Nottingham Forest']
}

ARTETA_SEASONS = ['1920', '2021', '2122', '2223', '2324', '2425', '2526']

TEAM_PALETTE = {
    'Arsenal':            '#EF0107',
    'Liverpool':          '#00B2A9',
    'Manchester City':    '#6CABDD',
    'Manchester United':  '#FFB81C',
}

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

print('Imports ready')

Imports ready


---
## Step 2 - Load Data

In [5]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROC_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(PROC_DATA_DIR / 'all_4teams_processed.csv')
df['date'] = pd.to_datetime(df['date'])
df['season'] = df['season'].astype(str)

print(f'Shape: {df.shape}')

Shape: (1064, 37)


---
## Section A - What a Monte Carlo Simulation Actually Is

A quick illustration of drawing random outcomes from a fixed probability triple, confirming the sampling mechanic behaves as expected before it gets used for real.

In [6]:
# a quick illustration: one match, simulated 10,000 times, using its own random draw each time
p_win, p_draw, p_loss = 0.60, 0.25, 0.15

outcomes = rng.choice(['Win', 'Draw', 'Loss'], size=10000, p=[p_win, p_draw, p_loss])
observed = pd.Series(outcomes).value_counts(normalize=True).reindex(['Win', 'Draw', 'Loss'])

print('Input probabilities:  Win 0.60, Draw 0.25, Loss 0.15')
print('Observed frequency across 10,000 independent draws:')
print(observed.round(3))

Input probabilities:  Win 0.60, Draw 0.25, Loss 0.15
Observed frequency across 10,000 independent draws:
Win     0.605
Draw    0.248
Loss    0.148
Name: proportion, dtype: float64


---
## Section B - The Live Data Problem

NB01's `fixtures_26-27.csv` / `all_teams_26-27_raw.csv` were scraped from FBRef with `xG`/`xGA` left blank and may be several gameweeks stale. Check their actual state directly rather than assuming a file called "raw" is current.

In [7]:
fx_old = pd.read_csv(RAW_DATA_DIR / 'all_teams_26-27_raw.csv')
print(f'Rows: {len(fx_old)}, played (non-null scored): {fx_old["scored"].notna().sum()}')
print(f'xG non-null count: {fx_old["xG"].notna().sum()}')
print(f'Max gameweek with a real result: {fx_old.loc[fx_old["scored"].notna(), "gameweek"].max()}')

Rows: 760, played (non-null scored): 20
xG non-null count: 0
Max gameweek with a real result: 1


In [8]:
u_check = sd.Understat(leagues='ENG-Premier League', seasons='2627', no_cache=True)
sched_check = u_check.read_schedule().reset_index()
sched_check['date'] = pd.to_datetime(sched_check['date'])
played = sched_check[sched_check['home_goals'].notna()]

print(f'Understat 2026-27 matches with a real result: {len(played)}')
print(f'Latest played date: {played["date"].max()}')
print()
print('A sample played match, home_xg/away_xg populated:')
print(played[['date','home_team','away_team','home_goals','away_goals','home_xg','away_xg']].tail(3).to_string(index=False))

[09/18/26 03:41:46] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=10693305;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=10693306;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-09-18 03:41:46] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=10693313;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=10693314;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\                 
                             bin\tls-client-xgo-1.13.1-windows-amd64.dll                                           

Understat 2026-27 matches with a real result: 40
Latest played date: 2026-09-14 19:00:00

A sample played match, home_xg/away_xg populated:
               date         home_team        away_team  home_goals  away_goals   home_xg   away_xg
2026-09-13 13:00:00          Coventry         Brighton           0           5   1.04812   2.76518
2026-09-13 15:30:00 Manchester United  Manchester City           0           1  0.943406   1.14073
2026-09-14 19:00:00             Leeds Newcastle United           4           1    2.2902  0.793165


---
## Section C - Building the Live Update

Pull the current 20-team schedule fresh from Understat (`no_cache=True`), reshape it into this project's usual team-per-row long format, and write it to new `fixtures_26-27_live.csv` / `all_teams_26-27_live.csv` files, leaving NB01's original output untouched.

In [17]:
def fetch_live_schedule(season='2627'):
    u = sd.Understat(leagues='ENG-Premier League', seasons = season, no_cache=True)
    sched = u.read_schedule().reset_index()
    sched['date'] = pd.to_datetime(sched['date'])

    home = sched.rename(columns={
        'home_team': 'team', 'away_team': 'opponent',
        'home_goals': 'scored', 'away_goals': 'conceded',
        'home_xg': 'xG', 'away_xg': 'xGA',        
    })
    home['venue'] = 'home'
    away = sched.rename(columns={
        'away_team': 'team', 'home_team': 'opponent',
        'away_goals': 'scored', 'home_goals': 'conceded',
        'away_xg': 'xG', 'home_xg': 'xGA',
    })
    away['venue'] = 'away'

    cols = ['league', 'season', 'game_id', 'date', 'team', 'opponent', 'venue', 'xG', 'xGA', 'scored', 'conceded']
    long = pd.concat([home[cols], away[cols]], ignore_index=True)
    long['gameweek'] = long.groupby('team')['date'].rank(method='dense').astype(int)
    return long.sort_values(['team','date']).reset_index(drop=True)

live_2627 = fetch_live_schedule()
print(f'Live 2026-27 long format: {live_2627.shape}')
print(f'Teams: {live_2627["team"].nunique()}')
print(f'Played rows: {live_2627["scored"].notna().sum()} / {len(live_2627)}')
print(f'Latest played gameweek: {live_2627.loc[live_2627["scored"].notna(), "gameweek"].max()}')

[09/18/26 03:50:21] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=10693319;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=10693320;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Live 2026-27 long format: (760, 12)
Teams: 20
Played rows: 80 / 760
Latest played gameweek: 4


In [18]:
all20_live = live_2627.copy()
tracked_live = live_2627[live_2627['team'].isin(TITLE_TEAMS)].copy()

all20_live.to_csv(RAW_DATA_DIR / 'all_teams_26-27_live.csv', index=False)
tracked_live.to_csv(RAW_DATA_DIR / 'fixtures_26-27_live.csv', index=False)

print(f'Wrote all_teams_26-27_live.csv : {len(all20_live)} rows')
print(f'Wrote fixtures_26-27_live.csv  : {len(tracked_live)} rows')

Wrote all_teams_26-27_live.csv : 760 rows
Wrote fixtures_26-27_live.csv  : 152 rows


---
## Section D - Scope: One Model, Not Two

A genuine title race needs all 20 teams, not just the 4 tracked here. Training a second, simpler model for the other 16 looks reasonable, check whether it introduces a population bias before committing to it.

In [23]:
# rebuild the two-model version first, to see the problem directly
hist = sd.Understat(leagues='ENG-Premier League', seasons=ARTETA_SEASONS)
hist_sched = hist.read_schedule().reset_index()
hist_sched['date'] = pd.to_datetime(hist_sched['date'])

def wide_to_long(sched):
    home = sched.rename(columns={'home_team':'team','away_team':'opponent','home_goals':'scored',
                                   'away_goals':'conceded','home_xg':'xG','away_xg':'xGA'})
    home['venue'] = 'home'
    away = sched.rename(columns={'away_team':'team','home_team':'opponent','away_goals':'scored',
                                   'home_goals':'conceded','away_xg':'xG','home_xg':'xGA'})
    away['venue'] = 'away'
    cols = ['league','season','game_id','date','team','opponent','venue','xG','xGA','scored','conceded']
    return pd.concat([home[cols], away[cols]], ignore_index=True).sort_values(['team','season','date']).reset_index(drop=True)

full_long = wide_to_long(hist_sched)
for c in ['scored','conceded','xG','xGA']:
    full_long[c] = full_long[c].astype('float64')

full_long['result'] = np.where(full_long['scored'] > full_long['conceded'], 'W',
    np.where(full_long['scored'] == full_long['conceded'], 'D','L'))
full_long['points'] = full_long['result'].map({'W':3,'D':1,'L':0})
full_long['xgd'] = full_long['xG']- full_long['xGA']
full_long['is_win'] = (full_long['result'] == 'W').astype(float)
full_long['is_home'] = (full_long['venue'] == 'home').astype(int)
full_long['is_big6_opp'] = full_long['opponent'].isin(BIG6).astype(int)

def roll(g, col, window, min_p= 3):
    return g[col].transform(lambda x: x.shift(1).rolling(window, min_periods=min_p).mean())
g = full_long.groupby(['team','season'])
full_long['xG_roll5'] = roll(g, 'xG',5)
full_long['xGA_roll5'] = roll(g, 'xGA', 5)
full_long['pts_roll5'] = roll(g, 'points', 5)
full_long['win_rate_roll5'] = roll(g, 'is_win', 5)
full_long['xgd_roll5'] = roll(g, 'xgd', 5)

opp_lookup = full_long[['team','season','game_id','xgd_roll5']].rename(columns={'team':'opponent','xgd_roll5':'opp_xgd_roll5'})
full_long = full_long.merge(opp_lookup, on = ['opponent','season','game_id'], how = 'left')
full_long['parity_gap'] = (full_long['xgd_roll5'] - full_long['opp_xgd_roll5']).abs()

print(f'Full 20-team long shape: {full_long.shape}')

[09/18/26 04:06:00] INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=10693343;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=10693344;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

Full 20-team long shape: (5320, 24)


In [26]:
FEATURES_T2 = ['xG_roll5','xGA_roll5','pts_roll5','win_rate_roll5','is_home','is_big6_opp','opp_xgd_roll5','parity_gap']

all_rows = full_long[full_long['season'].isin(ARTETA_SEASONS)].copy()
y_map = {'L':0,'D':1,'W':2}
all_rows['y'] = all_rows['result'].map(y_map)
t2_df = all_rows.dropna(subset = FEATURES_T2 + ['y']).copy()
t2_df['season_int'] = t2_df['season'].astype(int)
t2_train, t2_test = t2_df[t2_df['season_int']<=2425] , t2_df[t2_df['season_int'] == 2526]

sc2 = StandardScaler()
m2 = LogisticRegression(max_iter=2000).fit(sc2.fit_transform(t2_train[FEATURES_T2]), t2_train['y'])

avg_t2 = t2_train[FEATURES_T2].mean().to_frame().T
p_avg_t2 = m2.predict_proba(sc2.transform(avg_t2))[0]
print(f'Tier-2 model (all 20 teams), average team baseline -> L/D/W = {p_avg_t2.round(3)}')

Tier-2 model (all 20 teams), average team baseline -> L/D/W = [0.375 0.25  0.375]


In [29]:
FEATURES_T1 = ['xG_roll5','xGA_roll5','pts_roll5','win_rate_roll5','stakes_intensity','is_home',
               'is_big6_opp','is_non_big6_rivalry','opp_xgd_roll5']

t1_processed = pd.read_csv(PROC_DATA_DIR / 'all_4teams_processed.csv')
t1_processed['is_home'] = (t1_processed['venue'] == 'home')
t1_processed['y'] = t1_processed['result'].map(y_map)

# is_big6_opp / is_non_big6_rivalry / opp_xgd_roll5 aren't saved in the processed CSV,
# NB06 computed them inline, so this rebuilds the same 3 columns the same way
t1_processed['is_big6_opp'] = t1_processed['opponent'].isin(BIG6).astype(int)
t1_processed['is_non_big6_rivalry'] = (t1_processed['is_rivalry'] & (t1_processed['is_big6_opp']==0)).astype(int)
opp_lookup_t1 = full_long[['team','season','game_id','xgd_roll5']].rename(columns={'team':'opponent','xgd_roll5':'opp_xgd_roll5'})
t1_processed['season'] = t1_processed['season'].astype(str)
opp_lookup_t1['season'] = opp_lookup_t1['season'].astype(str)
t1_processed = t1_processed.merge(opp_lookup_t1, on=['opponent','season','game_id'], how='left')

t1_df = t1_processed.dropna(subset=FEATURES_T1 + ['y']).copy()
t1_df['season_int'] = t1_df['season'].astype(int)
t1_train = t1_df[t1_df['season_int']<=2425]

sc1 = StandardScaler()
m1 = LogisticRegression(max_iter=2000).fit(sc1.fit_transform(t1_train[FEATURES_T1]), t1_train['y'])
avg_t1 = t1_train[FEATURES_T1].mean().to_frame().T
p_avg_t1 = m1.predict_proba(sc1.transform(avg_t1))[0]
print(f'Tier-1 model (4 tracked teams), average team baseline -> L/D/W = {p_avg_t1.round(3)}')
print(f'Tier-2 model (all 20 teams),   average team baseline -> L/D/W = {p_avg_t2.round(3)}')
print(f'Win probability gap from model choice alone: {p_avg_t1[2]-p_avg_t2[2]:+.3f}')

Tier-1 model (4 tracked teams), average team baseline -> L/D/W = [0.181 0.213 0.606]
Tier-2 model (all 20 teams),   average team baseline -> L/D/W = [0.375 0.25  0.375]
Win probability gap from model choice alone: +0.232


---
## Section E - Stakes Intensity for All 20 Teams

`title_gap`, `cl_gap`, and `eur_gap` already generalise to all 20 teams from NB02's full-schedule reconstruction. Extend `stakes_intensity` beyond the 4 tracked teams, and add the relegation boundary the original formula never needed.

In [31]:
# reconstruct the full stakes_intensity ingredients for ALL 20 teams
full_long['gameweek'] = full_long.groupby(['team','season'])['date'].rank(method= 'dense').astype(int)
full_long['cum_pts'] = full_long.groupby(['team','season'])['points'].transform(lambda x: x.fillna(0).cumsum())
full_long['leader_pts'] = full_long.groupby(['season','gameweek'])['cum_pts'].transform('max')
full_long['title_gap'] = full_long['leader_pts'] - full_long['cum_pts']



---
## Section F - One More Real Signal: Head-to-Head Record

Test head-to-head record against the specific opponent, and whatever else seems worth trying, against the same holdout, keeping only what actually survives.

---
## Section G - The Production Model, and What "Today" Means

Refit the chosen recipe on everything through 2025-26 rather than holding it back as a permanent test set, then build each team's frozen form snapshot as of their next unplayed 2026-27 fixture.

---
## Section H - Simulating One Match, Then One Season

Sample a discrete outcome from a probability triple, then assemble the remaining fixtures with each one's model perspective, tracked teams needing dynamic `stakes_intensity`, the rest fixed for the season.

---
## Section I - The Simulation Engine

Run the gameweek-by-gameweek season simulator across many runs, then check convergence, whether the run count actually settled on an answer, rather than assuming a round number is enough.

---
## Section J - Does the Simulator Actually Work?

Freeze at a past gameweek of a real, completed season and simulate forward, checking whether the actual final table falls inside the simulated distribution.

---
## Key Findings Summary